Grundregeln:
- jede python Klasse hat eine übergeordnete Aufgabe
- jede python Klasse hält nur einen data file
- jede python Klasse importiert nur dann andere Klassen, wenn sie nur durch diese Importe als Gesamtes funktionieren (das ist in den meisten Fällen der DataManager). Wenn eine andere Klasse kein inhärenter Teil ist, wird sie nur als Funktions-Input für die Methode nutzbar gemacht.
- kein cleaning! Statt stillem Bereinigen stoppt die Funktion bei Ungereimtheiten und zeigt eine Fehlermeldung an

Zu adressierende Probleme:
- predicate requisite kleiner Teil der RO-Relationen. Reicht der aus?
- nur Abstracts in pubmed. Volltext-Reviews / Metaanalysen oder Bücher besser?
- manche Attribute sind bei fast jeder Entität TRUE. Sollte man diese entfernen? >90% TRUE?
- Scalability probleme vor allem der Normalisierung und fill_nan
- Vektordatenbank statt Pubmed_Abstracts_df besser?




Adressierte Probleme:
- causally_related ist oft falsch verknüpft. Aus der Liste der Prädikate entfernen? Antwort: Entfernt worden
- bis LLMClient().extract_entities werden nur Attribute ausgefüllt, die TRUE sind. Das ist zu ungenau. Von Anfang an müssen Attribute auch False gelabelt werden. Antwort: Extract entities labelt auch Falsche Entitäten
- mehrere Klassen haben durch das LLM denselben Namen bekommen, was verboten ist. Antwort: Prompt ist angepasst worden
- Abstracts enthalten sehr viele Entitäten, die nichts mit Wunden zu tun haben. Antwort: LLM prompt wurde so angepasst, dass nur domänenspezifische Entitäten integriert werden.
- Die formale Konzeptanalyse hält sich nicht an die BFO/OBO-Architektur. Antwort: attributes haben die form predicate: target. präformierte attributes sind die bfo_classes als is_a:bfo_class um die Einteilung in die BFo-Architektur zu erzwingen.
- es kann mehrere synonme Objekte oder Attribute geben. Antwort: objects und attributes werden per LLM zu standardbegriffen gemappt, dann grouping durch hard code
- Klassennamen werden aus Attributen gebildet. Antwort: Semantische Klassennamen kreiert LLM
- zu viele concepts und attribute? unschaffbar für hard code? erweiterter RAM und TPU
- zu viele Anfragen an LLM? Antwort: batching
- zu komplexe Anfragen an LLM? Antwort: Lange Frage besser als lange Antwort! (gemini 3 flash schafft 1mio tokens)
- komplette Nachverfolgbarkeit für selektives Löschen query-weise oder datumsweise bei schlechten einzelnen Ergebnissen: Antwort: meta_ids
- kein überladen des Speicher: Antwort: immer nur 10 dateien eines Präfix im Speicher und purge method des DataManager()
- LLM-Anfragen dauern zu lange, teils nur eine Anfrage pro 2,5 Minuten. Antwort: asyncio parallel prompting


# 1. Einrichtung der Grundbausteine

## 1.2 Installieren der benötigten Bibliotheken

## 1.3 Import der benötigten Bibliotheken

In [28]:
import nest_asyncio
nest_asyncio.apply()



In [29]:
import pandas as pd
import numpy as np
import re
import os
import glob
import seaborn as sns
from pydantic import validate_call, ConfigDict
import matplotlib.pyplot as plt
from scipy.stats import kruskal, chi2_contingency
import statsmodels
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
#from adjustText import adjust_text
import statsmodels.formula.api as smf

In [30]:
import pymc as pm
import arviz as az
import tabulate
import jax
import time
import json
import google.genai as genai
from concepts import Context
import owlready2
import time
from google.api_core import exceptions
from owlready2 import *
import asyncio
from Bio import Entrez, Medline
from IPython.display import display
import types
from abc import ABC, abstractmethod
import requests
from datetime import datetime
from pathlib import Path
from google.genai import types
from pydantic import BaseModel, Field
import pickle


from google.genai.errors import ServerError

In [31]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# Works whether VS Code runs the notebook from project root or notebooks/
_cwd = Path().resolve()
_project_root = _cwd if (_cwd / 'config').exists() else _cwd.parent
sys.path.insert(0, str(_project_root))
load_dotenv(_project_root / '.env')

from config.settings import (
    BASE_PATH,
    OWL_OUTPUT_FILE,
    BFO_OWL_URL,
    RO_OWL_URL,
    ENTREZ_EMAIL,
    NCBI_API_KEY,
    GOOGLE_API_KEY,
    BATCH_SIZE,
    CACHE_MAX_FILES_PER_PREFIX,
    PUBMED_SEARCH_TERMS,
    ALLOWED_PREDICATES,
)
print(f"Project root: {_project_root}")
print(f"ALLOWED_PREDICATES: {ALLOWED_PREDICATES}")

Project root: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Wound ontology
ALLOWED_PREDICATES: ['is_a', 'part_of', 'has_part', 'has_quality', 'occurs_in', 'has_participant']


# Klassenbibliothek

In [ ]:
from wound_ontology.data_manager.data_manager import DataManager


In [ ]:
from wound_ontology.connectors.base_connector import BaseConnector


In [ ]:
from wound_ontology.connectors.onto_connector import OntoConnector


In [ ]:
from wound_ontology.connectors.bfo_connector import BFOConnector


In [ ]:
from wound_ontology.connectors.ro_connector import ROConnector


In [ ]:
from wound_ontology.connectors.umls_connector import UMLSConnector


In [ ]:
from wound_ontology.connectors.pubmed_connector import PubmedConnector


In [ ]:
from wound_ontology.llm.schemas import FCAAttribute, Entity, ExtractResponse, NormalizationEntry, NormalizationResponse, FillNanResponse, SemanticNameResponse


In [ ]:
from wound_ontology.llm.llm_client import LLMClient


In [ ]:
from wound_ontology.cache.extraction_cache import ExtractionCache


In [ ]:
from wound_ontology.fca.matrix import _Matrix


In [ ]:
from wound_ontology.fca.formal_context import FormalContext


In [ ]:
from wound_ontology.mapping.base_mapper import BaseMapper


In [ ]:
from wound_ontology.mapping.normalization_mapper import NormalizationMapper

In [ ]:
from wound_ontology.mapping.target_mapper import TermMapper

In [ ]:
from wound_ontology.hierarchies.base_hierarchy import BaseHierarchy


In [ ]:
from wound_ontology.hierarchies.class_hierarchy import ClassHierarchy


In [ ]:
from wound_ontology.hierarchies.mereological_hierarchy import MereologicalHierarchy

In [ ]:
from wound_ontology.fca.concept_analysis import FormalConceptAnalysis

## class OWL Exporter

In [ ]:
from wound_ontology.export.owl_exporter import OWLExporter


## class Orchestrator

In [ ]:
from wound_ontology.orchestrator import Orchestrator


# Pipeline

In [67]:
pipeline = Orchestrator()
#pipeline.dm.purge_by_prefix("context_matrix",confirm=True)
#pipeline.run_full_pipeline(PUBMED_SEARCH_TERMS[0])
#pipeline.run_extract_to_final_cache()
pipeline.run_matrix_nan_fill()
pipeline.run_fca()
pipeline.export_to_owl()

Directory ready: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Wound ontology/data/cache
Loading latest JSON: meta_ids_2026-04-08_23-00.json
Initializing PubMed API...
Loading latest CSV: pubmed_abstracts_2026-04-08_23-00.csv
Connected to PubMed via moritz.milewski@master-bids.de
No CSV found for prefix: context_matrix
Loading latest JSON: extraction_cache_2026-04-08_23-15.json
No Pickle found for prefix: concept_lattice
No JSON found for prefix: target_mapping
No JSON found for prefix: object_mapping
Synchronizing Matrix...
✅ Sparse Sync Successful (1.0=True, 0.0=False, NaN=Unknown).
   - Unique Entities: 328
   - Medical Triples: 661
Creating normalization mappings (Parallel)...
[Mapper of the target terms in matrix attributes] Found 653 new terms. Processing 22 async chunks...
[Mapper of the target terms in matrix attributes] Chunk 9 complete.
[Mapper of the target terms in matrix attributes] Saving updated mapping...
[Mapper of the target terms in matrix attributes

KeyboardInterrupt: 

In [ ]:
pipeline.lattice.summary()